In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [6]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
2,model_29_6_0,0.780208,-0.056120,0.447242,0.937859,0.918384,0.359912,1.729406,0.238218,0.383869,0.311043,0.839294,0.599926,1.586112,0.625466,68.043794,108.266696,"Hidden Size=[8], regularizer=0.2, learning_rat..."
4,model_29_6_1,0.780020,-0.050951,0.423417,0.934040,0.913942,0.360220,1.720942,0.248486,0.407459,0.327972,0.862678,0.600183,1.586614,0.625734,68.042082,108.264984,"Hidden Size=[8], regularizer=0.2, learning_rat..."
5,model_29_6_2,0.779565,-0.045940,0.398449,0.929961,0.909224,0.360965,1.712737,0.259246,0.432657,0.345951,0.886442,0.600803,1.587827,0.626381,68.037951,108.260853,"Hidden Size=[8], regularizer=0.2, learning_rat..."
7,model_29_6_3,0.778838,-0.041092,0.372386,0.925607,0.904222,0.362155,1.704798,0.270478,0.459551,0.365014,0.910512,0.601793,1.589765,0.627413,68.031367,108.254269,"Hidden Size=[8], regularizer=0.2, learning_rat..."
8,model_29_6_4,0.777834,-0.036411,0.345263,0.920965,0.898927,0.363800,1.697133,0.282167,0.488225,0.385196,0.934819,0.603158,1.592444,0.628836,68.022304,108.245207,"Hidden Size=[8], regularizer=0.2, learning_rat..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430,model_15_5_5,0.677246,0.048574,0.429106,0.623236,0.501646,0.528513,1.557968,0.794663,0.081577,0.438120,0.810092,0.726989,1.267107,0.757939,107.275374,171.875793,"Hidden Size=[13], regularizer=0.05, learning_r..."
431,model_25_9_0,0.676118,0.055705,0.894964,0.275929,0.883456,0.530360,1.546291,0.689249,0.200302,0.444775,0.922220,0.728258,2.554635,0.759262,59.268397,94.615796,"Hidden Size=[7], regularizer=0.2, learning_rat..."
432,model_15_4_18,0.675558,0.057990,0.600245,0.637987,0.614386,0.531276,1.542550,2.476681,0.396339,1.436510,0.828303,0.728887,1.268503,0.759917,107.264946,171.865365,"Hidden Size=[13], regularizer=0.05, learning_r..."
433,model_15_5_4,0.675474,0.049762,0.447981,0.645115,0.519283,0.531414,1.556024,0.768390,0.076840,0.422615,0.814459,0.728982,1.268573,0.760016,107.264426,171.864845,"Hidden Size=[13], regularizer=0.05, learning_r..."
